# w9_packageview.ipynb — N 包分形 i2ce（pk 家族）

User design: 在 pk（多个锚包）上做 i2ce，子视图上做 CE 或 i2ce。Each of the PKN
packs is an INDEPENDENT **seeded random h-sample** of the game's anchor
sentences (fixed per-game Generator seed, bit-reproducible; packs may
OVERLAP — user rejected disjoint contiguous slices). Short pools wrap via
tiled randperm, so packs are always full; the pack UNION stays inside the
same cap-sized anchor set as the joint-pool tower at equal cap. Pack level: per-pack CE vs the alive-weighted mean gallery
(anti-gaming lever; `pk2i2vce` drops it = pack-I only) + pack-I x2 over pack
pairs. Views: `vce` CE only / `vi2ce` full i2ce (view-I x2) / `sgvce`
detached gallery. Eval gallery = normalize(alive-mean(e_1..e_PKN)).

WAVE 2 (user 2026-07-18), three towers. **User notation: pk@N = PER-PACK
size** (worker --anchor-cap = total = PKN x N):
**pk2i2vce@512** (2x512 = g1024 -- pack-I without pack-CE, single-factor
pair vs the done vce tower), **pk4i2cevi2ce@512** (4x512 = g2048 -- same
pack size as the champion, pack COUNT 2->4), **pk2i2cevi2ce@2048** (2x2048
= g4096 -- same TOTAL budget as the i2ce@4096 joint tower: can split packs
rival it?). All packs = seeded random samples (overlap allowed) of the
same cap-deterministic anchor set. Dropped cell: 2x1024 (g2048 pk2).
~13.5G (g1024) / ~22G (g2048) / ~45G (g4096).
ZS-only, rvsel selection. Needs worker >= the N-pack commit. AUTO-STOPS.


In [ ]:
# constants
import os

REPO = os.path.abspath("..")   # this release folder (contains Pod/ and VICReg_review/)
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_out"          # fixed-split campaign dir

# WAVE 2 (user 2026-07-18): exactly three towers, everything else dropped.
# USER NOTATION: pk@N = PER-PACK size; worker --anchor-cap = total = PKN*N.
JOBS = [("wcle_pk2i2vce_icetf", 1024),      # pk2@512: 2x512, pack-I only
        ("wcle_pk4i2cevi2ce_icetf", 2048),  # pk4@512: 4x512 -- more packs, champion pack size
        ("wcle_pk2i2cevi2ce_icetf", 4096)]  # pk2@2048: 2x2048 -- same TOTAL as i2ce@4096
EPOCHS = 2000
os.makedirs(OUT_DIR, exist_ok=True)
print("towers:", [f"w9_{a}_g{c}" for a, c in JOBS], f"@ {EPOCHS}ep")


In [ ]:
# Local setup (release build: the code ships with this folder -- no
# repository synchronisation is needed or performed).
import importlib.util
import os
import sys
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        %pip -q install scikit-learn scipy
        break
os.chdir(REPO)
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")

In [ ]:
# Stage the corpus into RAM (llm views not needed).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)


In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# Run the pending towers (round-robin over GPUs; several fit per 80G card).
# ZS-only done marker = ep{EPOCHS} npz.
import os, subprocess, threading, time
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
gpus = J.detect_gpus()
todo = []
for arm, cap in JOBS:
    nm = J.fs_label(arm, cap, False, 0, "clean", 16)
    if (Path(OUT_DIR) / f"tower_{nm}_fp_ep{EPOCHS}.npz").exists():
        print(f"[skip] {nm} done"); continue
    todo.append((arm, cap, nm))
print(f"{len(todo)} tower(s) to run")

stop_evt = threading.Event()
threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True).start()
fails = []

def run_one(g, arm, cap, nm):
    if not J.try_claim(cdir, nm):
        # corpse window: a pod that died <120s ago still looks alive.
        # Wait out DEAD_SEC once and retry before giving up (fast relaunch
        # otherwise skips everything and auto-stops -- looks like a crash).
        print(f"[claim] {nm} fresh/held -- waiting 130s for the corpse "
              "window, then retrying once", flush=True)
        time.sleep(130)
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held elsewhere -- skipped", flush=True); return
    cmd = ["python", "-u", J.FS_WORKER, "--data-dir", DATA_DIR, "--out-dir",
           OUT_DIR, "--repo", REPO, "--arm", arm, "--anchor-cap", str(cap),
           "--epochs", str(EPOCHS), "--ckpt-every", str(J.CKPT_EVERY),
           "--ckpt-seeds", str(J.FS_CKPT_SEEDS),
           "--topup-seeds", str(J.TOPUP_SEEDS),
           "--full-pool", "--full-pool-path", FULL_POOL_PATH,
           "--claim-file", str(cdir / f"{nm}.claim")]
    print(f"[gpu{g}] start {nm}", flush=True)
    t0 = time.time()
    with open(logd / f"{arm}_g{cap}.log", "w") as fh:
        p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                           env=dict(os.environ, CUDA_VISIBLE_DEVICES=g))
    if p.returncode != 0:
        (cdir / f"{nm}.claim").unlink(missing_ok=True); fails.append(nm)
    print(f"[gpu{g}] {'ok' if p.returncode == 0 else 'FAIL'} {nm} "
          f"[{(time.time() - t0) / 3600:.1f}h]", flush=True)

ths = [threading.Thread(target=run_one, args=(gpus[i % len(gpus)], arm, cap, nm))
       for i, (arm, cap, nm) in enumerate(todo)]
for i, t in enumerate(ths):
    if i:
        time.sleep(120)   # stagger starts: each worker's data-load phase has
        # an ~8.5G host-RAM transient (POOL np.load) + array loads; three
        # simultaneous startups stack the peak -- suspect in the container-OOM
        # pod deaths (attempt1 died at ~ep50, right after triple load).
    t.start()
for t in ths:
    t.join()
stop_evt.set()
print(f"done; {len(fails)} failed")
for nm in fails:
    print("  FAILED:", nm)


In [ ]:
# Readout: twin-pack pair vs the joint-pool references (ZSbest-primary).
import json
import numpy as np
from pathlib import Path
VORD = ["neutral", "noname", "positive", "negative"]
ROWS = [("wcle_pk2i2vce_icetf_g1024", "pk2i2vce 2x512 (I-only)"),
        ("wcle_pk4i2cevi2ce_icetf_g2048", "pk4i2cevi2ce 4x512"),
        ("wcle_pk2i2cevi2ce_icetf_g4096", "pk2i2cevi2ce 2x2048"),
        ("wcle_pk2i2cevce_icetf_g1024", "pk2i2cevce 2x512 (vce)"),
        ("wcle_pk2i2cevi2ce_icetf_g1024", "pk2i2cevi2ce 2x512 (champ)"),
        ("wcle_pk2i2cesgvce_icetf_g1024", "pk2i2cesgvce 2x512 (sg)"),
        ("wcle_i2ce_icetf", "i2ce@512 (joint ref)"),
        ("wcle_ce_cetf", "ce@512 (ref)"),
        ("wcle_i2ce_icetf_g1024", "i2ce@1024 (joint ref)"),
        ("wcle_i2ce_icetf_g2048", "i2ce@2048 (joint ref)"),
        ("wcle_i2ce_icetf_g4096", "i2ce@4096 (pk2 2x2048 target)")]
def _row(lab, nm):
    zb = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
    zp = Path(OUT_DIR) / f"zs_traj_{nm}_fp.json"
    ft = Path(OUT_DIR) / f"ft4var_{nm}_fp_best.json"
    if zb.exists():
        d = json.loads(zb.read_text())
        m4z = np.mean([d["nm_" + v] for v in VORD])
        line = (f"{lab:30s} ZSbest@ep{d['best_ep']:>4}(val) "
                + " ".join(f"{v[:3]}:{d['nm_' + v]:.3f}" for v in VORD)
                + f" m4z:{m4z:.3f} tag:{d['tag_neutral']:.3f}/{d['tag_noname']:.3f}")
    elif zp.exists():
        tr = json.loads(zp.read_text())
        eps = sorted(tr, key=lambda k: int(k[2:]))
        pk = max(eps, key=lambda k: tr[k]["nm_neutral"])
        line = (f"{lab:30s} ZS test-peak*@{pk[2:]:>4} neu {tr[pk]['nm_neutral']:.3f}"
                f" non {tr[pk]['nm_noname']:.3f}"
                f" tag {tr[pk]['tag_neutral']:.3f}/{tr[pk]['tag_noname']:.3f}")
    else:
        return f"{lab:30s} (pending)"
    if ft.exists():
        d2 = json.loads(ft.read_text())
        m4 = np.mean([np.mean([x[v]["h1"] for x in d2["per_seed"]]) for v in VORD])
        line += f" | FT m4 {m4:.3f}"
    return line

for arm, lab in ROWS:
    print(_row(lab, f"w9_{arm}"))


In [ ]:
# AUTO-STOP removed in the release build: stopping the machine is cloud-
# provider tooling, not part of the experiment. All results are already on
# the shared volume when the run cells finish.
print("run complete -- results are in", OUT_DIR)